In [324]:
%pip install ipython-sql
# %sql SELECT name FROM sqlite_master WHERE type='table'

Note: you may need to restart the kernel to use updated packages.


In [325]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [326]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [367]:
import numpy as np
import pandas as pd

In [368]:
chemin = r"C:\Users\USER\Desktop\Learning Data Analyst\DataRockStar\Projet Fil Rouge\mon_projet\data\demographie-secteurs-conventionnels.csv"
df1 = pd.read_csv(chemin, sep=";")

In [369]:
df1.head()

,﻿annee,profession_sante,region,libelle_region,departement,libelle_departement,secteur_conventionnel,libelle_secteur_conventionnel,effectif,vision generale all,vision_generale_prescriptions,vision profession territoire
0,2024,Allergologues,1,Guadeloupe,971,Guadeloupe,3,conventionnés de secteur 2 ayant adhéré à l'Op...,0,non,oui,oui
1,2024,Allergologues,1,Guadeloupe,971,Guadeloupe,5,non conventionnés,0,non,oui,oui
2,2024,Allergologues,1,Guadeloupe,999,Tout département,3,conventionnés de secteur 2 ayant adhéré à l'Op...,0,non,oui,oui
3,2024,Allergologues,1,Guadeloupe,999,Tout département,4,conventionnés de secteur 2 n'ayant pas adhéré ...,0,non,oui,oui
4,2024,Allergologues,2,Martinique,972,Martinique,3,conventionnés de secteur 2 ayant adhéré à l'Op...,0,non,oui,oui


In [370]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 177720 entries, 0 to 177719
Data columns (total 12 columns):
 #   Column                         Non-Null Count   Dtype
---  ------                         --------------   -----
 0   ﻿annee                         177720 non-null  int64
 1   profession_sante               177720 non-null  str  
 2   region                         177720 non-null  int64
 3   libelle_region                 177720 non-null  str  
 4   departement                    177720 non-null  str  
 5   libelle_departement            177720 non-null  str  
 6   secteur_conventionnel          177720 non-null  int64
 7   libelle_secteur_conventionnel  177720 non-null  str  
 8   effectif                       177720 non-null  int64
 9   vision generale all            177720 non-null  str  
 10  vision_generale_prescriptions  177720 non-null  str  
 11  vision profession territoire   177720 non-null  str  
dtypes: int64(4), str(8)
memory usage: 33.2 MB


In [371]:
df1 = df1.rename(columns={
    "﻿annee": 'annee',
    "departement": "code_dep"
})
df1 = df1.drop(columns=["region", "vision generale all", "vision_generale_prescriptions", "vision profession territoire"])
df1["secteur_conventionnel"] = df1["secteur_conventionnel"].astype(str)
df1 = df1[df1["profession_sante"].isin(["Médecins généralistes (hors médecins à expertise particulière - MEP)", "Médecins généralistes à expertise particulière (MEP)"])].reset_index(drop=True)
df1 = df1.replace({"departement" : {"999" : np.nan},
                    "annee" : {2024 : np.nan}}).dropna()
df1.replace({'code_dep': {"1": "01",
                        "2": "02",
                        "3": "03",
                        "4": "04",
                        "5": "05",
                        "6": "06",
                        "7": "07",
                        "8": "08",
                        "9": "09"}})
df1 = df1.replace({'libelle_secteur_conventionnel': {
    "conventionnés de secteur 2 ayant adhéré à l'Optam/Optam-CO": "conventionnés de secteur 2",
    "conventionnés de secteur 2 n'ayant pas adhéré à l'Optam/Optam-CO": "conventionnés de secteur 2"
}})

In [374]:
df1.info()

<class 'pandas.DataFrame'>
Index: 12720 entries, 307 to 13679
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   annee                          12720 non-null  float64
 1   profession_sante               12720 non-null  str    
 2   libelle_region                 12720 non-null  str    
 3   code_dep                       12720 non-null  str    
 4   libelle_departement            12720 non-null  str    
 5   secteur_conventionnel          12720 non-null  str    
 6   libelle_secteur_conventionnel  12720 non-null  str    
 7   effectif                       12720 non-null  int64  
dtypes: float64(1), int64(1), str(6)
memory usage: 2.3 MB


In [ ]:
# Créer une ligne avec un total
df1_total = df1.groupby(["code_dep", "annee"], as_index=False)["effectif"].sum()

# Ajout d'une valeur spécifique dans les colonnes où total devra apparaître
df1_total["profession_sante"] = "Tous médecins généralistes"
df1_total["libelle_secteur_conventionnel"] = "tout secteur compris"
df1_total["secteur_conventionnel"] = "_T"

libelles = df1[["code_dep", "libelle_region", "libelle_departement"]].drop_duplicates()
df1_total = pd.merge(df1_total, libelles, on="code_dep", how="left")
# pd.concat() s'applique sur : liste de DataFrames → retourne un DataFrame
df1 = pd.concat([df1, df1_total], ignore_index=True)

In [376]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 14400 entries, 0 to 14399
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   annee                          14400 non-null  float64
 1   profession_sante               14400 non-null  str    
 2   libelle_region                 14400 non-null  str    
 3   code_dep                       14400 non-null  str    
 4   libelle_departement            14400 non-null  str    
 5   secteur_conventionnel          14400 non-null  str    
 6   libelle_secteur_conventionnel  14400 non-null  str    
 7   effectif                       14400 non-null  int64  
dtypes: float64(1), int64(1), str(6)
memory usage: 2.4 MB


In [ ]:
ok_year


[2023,
 2013,
 2020,
 2010,
 2017,
 2018,
 2019,
 2021,
 2016,
 2015,
 2011,
 2014,
 2012,
 2022]

In [ ]:
#df min/max => 2010/2024
df2 = pd.read_csv(r"C:\Users\USER\Desktop\Learning Data Analyst\DataRockStar\Projet Fil Rouge\mon_projet\data\DS_POPULATIONS_HISTORIQUES_data.csv", sep=";")


C:\Users\USER\AppData\Local\Temp\ipykernel_9432\609684049.py:3: DtypeWarning: Columns (0: GEO) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(chemin, sep=";")


In [ ]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 813234 entries, 0 to 813233
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   FREQ            813234 non-null  str   
 1   GEO             813234 non-null  object
 2   GEO_OBJECT      813234 non-null  str   
 3   POPREF_MEASURE  813234 non-null  str   
 4   TIME_PERIOD     813234 non-null  int64 
 5   OBS_VALUE       813234 non-null  int64 
dtypes: int64(2), object(1), str(3)
memory usage: 43.4+ MB


In [ ]:
# Index(['FREQ', 'GEO', 'GEO_OBJECT', 'POPREF_MEASURE', 'TIME_PERIOD', 'OBS_VALUE'])
# année min/max => 2010/2024
# Je supprime toutes les lignes inutiles pour ne garder que les lignes avec les années souhaitées (entre 2010 et 2024 incluses) 
# et les lignes traitant des départements
df2 = df2[df2["TIME_PERIOD"].isin(ok_year)]
df2 = df2[df2["GEO_OBJECT"] == "DEP"].reset_index(drop=True)
# Je supprimes les colonnes inutiles
df2 = df2.drop(columns=["FREQ", "GEO_OBJECT", "POPREF_MEASURE"])
# Je standardise les noms de colonnes
df2 = df2.rename(columns={
    "GEO": "code_dep",
    "TIME_PERIOD": "annee"
})
df2.columns = df2.columns.str.lower()
df2.info()


<class 'pandas.DataFrame'>
RangeIndex: 1400 entries, 0 to 1399
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   code_dep   1400 non-null   object
 1   annee      1400 non-null   int64 
 2   obs_value  1400 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 32.9+ KB


In [ ]:
for dep in df1["code_dep"].value_counts().index:
    if dep not in df2["code_dep"].value_counts().index:
        print(dep, "n'est pas dans df2['GEO']")
# 976 n'est pas dans df2['GEO']

976 n'est pas dans df2['GEO']


In [ ]:
for year in df1["annee"].value_counts().index:
    if year not in df2["annee"].value_counts().index:
        print(year, "n'est pas dans df2")
# 2024 n'est pas dans df2

2024 n'est pas dans df2


In [ ]:
df_densite_medecin = pd.merge(
    df1,
    df2,
    on=["code_dep", "annee"],
    how="inner"
)
# Attention, un département aura 8 lignes + 1 total pour une annee dans df1, mais 1 ligne pour une annee dans df2.

In [ ]:
df_densite_medecin.head()

,annee,profession_sante,region,libelle_region,code_dep,libelle_departement,secteur_conventionnel,libelle_secteur_conventionnel,effectif,obs_value
0,2023,Médecins généralistes (hors médecins à experti...,02,Martinique,972,Martinique,1,conventionnés de secteur 1,289,360630
1,2023,Médecins généralistes (hors médecins à experti...,03,Guyane,973,Guyane,3,conventionnés de secteur 2,2,293996
2,2023,Médecins généralistes (hors médecins à experti...,03,Guyane,973,Guyane,4,conventionnés de secteur 2,0,293996
3,2023,Médecins généralistes (hors médecins à experti...,04,La Réunion,974,La Réunion,4,conventionnés de secteur 2,3,889679
4,2023,Médecins généralistes (hors médecins à experti...,11,Ile-de-France,75,Paris,1,conventionnés de secteur 1,1305,2103778


In [379]:
df3 = pd.read_csv(r"C:\Users\USER\Desktop\Learning Data Analyst\DataRockStar\Projet Fil Rouge\mon_projet\data\DS_DECES_MORTALITE_SERIES_data.csv", sep=";")
df3.head()

,GEO,GEO_OBJECT,FREQ,AGE,EC_MEASURE,UNIT_MEASURE,SEASONAL_ADJUST,SEX,UNIT_MULT,OBS_STATUS,OBS_STATUS_FR,DECIMALS,TIME_PERIOD,OBS_VALUE
0,75,DEP,A,_T,DTH_PLACE_RES,NR,N,_T,0,A,D,0,2024,14079.0
1,75,DEP,A,_T,DTH_PLACE_RES,NR,N,_T,0,A,D,0,2013,13939.0
2,85,DEP,A,_T,DTH_PLACE_RES,NR,N,_T,0,A,D,0,2024,8126.0
3,66,DEP,A,_T,DTH_PLACE_RES,NR,N,_T,0,A,D,0,1985,4326.0
4,66,DEP,A,_T,DTH_PLACE_RES,NR,N,_T,0,A,D,0,1984,4220.0


In [380]:
# lecture des métadonnées de décès, et choix de ce que je garde
chemin3 = r"C:\Users\USER\Desktop\Learning Data Analyst\DataRockStar\Projet Fil Rouge\data utiles au projet\DS_DECES_MORTALITE_SERIES_CSV_FR\DS_DECES_MORTALITE_SERIES_metadata.csv"
df3_meta = pd.read_csv(chemin3, sep=";")
to_keep = ["OBS_VALUE", "DECIMALS","EC_MEASURE", "UNIT_MEASURE", "SEX", "AGE", "OBS_STATUS", "OBS_STATUS_FR"]
df3_meta = df3_meta[df3_meta["COD_VAR"].isin(to_keep)].reset_index(drop=True)
df3_meta = df3_meta.drop(columns="GEO_OBJECT")
df3_meta


,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD
0,AGE,Âge,Y0,0 an
1,AGE,Âge,Y5T9,De 5 à 9 ans
2,AGE,Âge,Y55T59,De 55 à 59 ans
3,AGE,Âge,Y40T44,De 40 à 44 ans
4,AGE,Âge,Y25T29,De 25 à 29 ans
5,AGE,Âge,Y15T19,De 15 à 19 ans
6,AGE,Âge,Y10T14,De 10 à 14 ans
7,AGE,Âge,Y_LT1,Moins de 1 an
8,AGE,Âge,Y45T49,De 45 à 49 ans
9,AGE,Âge,Y80T89,De 80 à 89 ans


In [381]:
code_age = df3_meta[df3_meta["COD_VAR"] == "AGE"][["COD_VAR", "LIB_VAR","COD_MOD", "LIB_MOD"]]
for ca in code_age.COD_MOD.values:
    if ca not in df3["AGE"].values:
        code_age["COD_MOD"] = code_age["COD_MOD"].replace(ca, np.nan)
code_age = code_age.dropna().reset_index(drop=True)
keepers = pd.concat([(df3_meta.iloc[[26, 30, 33, 36, 38, 39, 40]]), (code_age.iloc[[1, 5, 6, 7]])], axis=0)
keepers

,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD
26,DECIMALS,Décimales,0,Zéro
30,EC_MEASURE,Mesure état civil,LEXPEC,Espérance de vie
33,EC_MEASURE,Mesure état civil,MOR_STANDARD_RT,Taux de mortalité standardisé
36,OBS_STATUS,Statut de l'observation,A,Normale
38,OBS_STATUS,Statut de l'observation,R,Valeur révisée
39,OBS_STATUS_FR,Version,D,Définitif
40,OBS_STATUS_FR,Version,PROV,Provisoire
1,AGE,Âge,Y5T9,De 5 à 9 ans
5,AGE,Âge,Y15T19,De 15 à 19 ans
6,AGE,Âge,Y10T14,De 10 à 14 ans


In [382]:
# Suppression des lignes inutiles pour mes analyses cf keepers
# Suppression des lignes qui ne contiennent pas de données départementales
# Suppression des lignes hors des années étudiées (2010 à 2024)
df3 = df3[
    (df3["GEO_OBJECT"] == "DEP") 
    & (df3["FREQ"] == "A")
    & (df3["OBS_STATUS"] == "A")
    & (df3["OBS_STATUS_FR"] == "D")
    & (df3["EC_MEASURE"].isin(["DTH_PLACE_RES", "MOR_STANDARD_RT"]))
    & (df3["AGE"].isin(["Y_LT1", "Y_LT65", "Y_GE65", "_T"]))]
df3["TIME_PERIOD"] = df3["TIME_PERIOD"].astype(int)
df3 = df3[(df3["TIME_PERIOD"].isin(ok_year))]

df3 = df3.rename(columns={
    "GEO": "code_dep",
    "TIME_PERIOD": "annee"
})
df3.columns = df3.columns.str.lower()

In [383]:
# Suppression des colonnes contenant la même valeur unique
for col in df3.columns:
    if len(df3[col].unique()) == 1:
        df3 = df3.drop(columns=col)

In [384]:
df3.info()

<class 'pandas.DataFrame'>
Index: 3756 entries, 1 to 77361
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   code_dep      3756 non-null   str    
 1   age           3756 non-null   str    
 2   ec_measure    3756 non-null   str    
 3   unit_measure  3756 non-null   str    
 4   decimals      3756 non-null   int64  
 5   annee         3756 non-null   int64  
 6   obs_value     3756 non-null   float64
dtypes: float64(1), int64(2), str(4)
memory usage: 320.2 KB


In [385]:
df3.head(3)

,code_dep,age,ec_measure,unit_measure,decimals,annee,obs_value
1,75,_T,DTH_PLACE_RES,NR,0,2013,13939.0
7,75,_T,DTH_PLACE_RES,NR,0,2011,13771.0
11,78,_T,DTH_PLACE_RES,NR,0,2011,8430.0


In [386]:
df2.columns

Index(['FREQ', 'GEO', 'GEO_OBJECT', 'POPREF_MEASURE', 'TIME_PERIOD',
       'OBS_VALUE'],
      dtype='str')

In [387]:
import plotly.express as px
import requests

# --- 1. Récupérer le GeoJSON des départements français ---
geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"
geojson_dept = requests.get(geojson_url).json()

# --- 2. Ta carte choroplèthe ---
fig = px.choropleth(
    df2,                          # ton dataframe
    geojson=geojson_dept,        # les formes géographiques
    locations="code_dep", # colonne avec les codes département (01, 02... 974)
    featureidkey="properties.code", # clé dans le GeoJSON qui correspond à tes codes
    color="obs_value",      # colonne population → détermine la couleur
    color_continuous_scale="Blues", # dégradé de couleurs (tu peux changer)
    scope="europe",              # zoom sur l'Europe (inclut la France métro)
    labels={"obs_values": "Population"} # légende lisible
)

# --- 3. Recentrer la carte sur la France ---
fig.update_geos(
    center={"lat": 46.5, "lon": 2.5},
    projection_scale=6 # zoom (augmente pour zoomer davantage)
)

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

# fig.show()

ValueError: Value of 'locations' is not the name of a column in 'data_frame'. Expected one of ['FREQ', 'GEO', 'GEO_OBJECT', 'POPREF_MEASURE', 'TIME_PERIOD', 'OBS_VALUE'] but received: code_dep